# Day 44：MCP（Model Context Protocol）与智能体双层记忆架构

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 44）！今天我们将实战演示基于 **JSON-RPC 2.0** 的 `MCPServer` 与 `MCPClient`（支持 `initialize`、`tools/list`、`tools/call`、`resources/list`、`resources/read`），以及 **DualMemoryManager 双层记忆架构**（短期滑动窗口 + SQLite 长期键值/语义记忆）（`v0.4.0-day44`）。

## 1. 环境初始化与导入 `project_iterations/day_44` 模块
定位仓库根目录并将 `project_iterations/day_44` 加入 `sys.path`。

In [1]:
import json
import sys
import tempfile
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day44_dir = repo_root / "project_iterations" / "day_44"
if str(day44_dir) not in sys.path:
    sys.path.insert(0, str(day44_dir))

from zhishu_agent import (
    DualMemoryManager,
    MCPClient,
    MCPClientError,
    MCPServer,
    create_memory_enabled_mcp_server,
    make_jsonrpc_request,
)

print("已加载模块路径:", day44_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_44


## 2. JSON-RPC 2.0 `MCPServer` 与 `MCPClient` 基础交互
注册自定义工具与只读资源，完成 `initialize`、`tools/list`、`tools/call`、`resources/list`、`resources/read` 以及错误码校验。

In [2]:
server = MCPServer(name="demo-mcp-server", version="0.4.0-day44")
server.register_tool(
    name="calc_token_budget",
    description="估算上下文 Token 预算",
    input_schema={
        "type": "object",
        "properties": {"turns": {"type": "integer"}, "avg_tokens": {"type": "integer"}},
        "required": ["turns", "avg_tokens"],
    },
    handler=lambda args: {"total_tokens": args["turns"] * args["avg_tokens"]},
)
server.register_resource(
    uri="config://mcp/spec",
    name="MCP 协议说明",
    description="JSON-RPC 2.0 传输规范",
    content_or_reader="MCP uses JSON-RPC 2.0 for stateful capability negotiation.",
)

client = MCPClient(server=server)
init_info = client.initialize(client_name="notebook-host")
print("握手成功:", init_info["serverInfo"], "协议:", init_info["protocolVersion"])
print("工具列表:", [t["name"] for t in client.list_tools()])
print("调用结果:", client.call_tool("calc_token_budget", {"turns": 6, "avg_tokens": 350})["content"][0]["text"])
print("资源列表:", [r["uri"] for r in client.list_resources()])
print("资源内容:", client.read_resource("config://mcp/spec")["contents"][0]["text"])

# 验证未知方法返回 JSON-RPC -32601
err_resp = server.handle_message(make_jsonrpc_request("unknown/op", request_id=99))
print("未知方法错误码:", err_resp["error"]["code"], err_resp["error"]["message"])

握手成功: {'name': 'demo-mcp-server', 'version': '0.4.0-day44'} 协议: 2024-11-05
工具列表: ['calc_token_budget']
调用结果: {"total_tokens": 2100}
资源列表: ['config://mcp/spec']
资源内容: MCP uses JSON-RPC 2.0 for stateful capability negotiation.
未知方法错误码: -32601 未知方法: unknown/op


## 3. `DualMemoryManager`：短期滑动窗口与 SQLite 长期记忆
演示滑动窗口自动淘汰旧轮次，以及长期记忆跨实例持久化写入、精确召回与相似度检索。

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / "nb_memory.db"
    mem1 = DualMemoryManager(db_path=db_path, window_size=2)

    mem1.add_turn("sess-101", "user", "第1轮：你好")
    mem1.add_turn("sess-101", "assistant", "第1轮：你好，请问有什么可以帮您？")
    mem1.add_turn("sess-101", "user", "第2轮：查一下部署架构")
    print("短期滑动窗口 (maxlen=2):", [t["content"] for t in mem1.get_short_term("sess-101")])

    mem1.remember("arch_stack", "智枢后端使用 Python 3.11 + FastAPI + SQLite", namespace="prod", tags=["架构", "python"])
    mem1.remember("sla_target", "核心检索接口 P99 延迟 <= 150ms", namespace="prod", tags=["sla", "性能"])

    # 模拟进程重启后重新连接 SQLite
    mem2 = DualMemoryManager(db_path=db_path, window_size=2)
    print("重启后精确召回 arch_stack:", mem2.recall_key("arch_stack", namespace="prod")["value"])
    hits = mem2.search_long_term("FastAPI Python 架构", namespace="prod", top_k=2)
    print("相似度检索 Top-1:", hits[0]["key"], "score=", hits[0]["score"])

短期滑动窗口 (maxlen=2): ['第1轮：你好，请问有什么可以帮您？', '第2轮：查一下部署架构']
重启后精确召回 arch_stack: 智枢后端使用 Python 3.11 + FastAPI + SQLite
相似度检索 Top-1: arch_stack score= 0.7071


## 4. 端到端闭环：记忆赋能的 MCP Server (`create_memory_enabled_mcp_server`)
通过 `MCPClient` 调用 `memory_store`、`memory_search` 并读取 `memory://stats` 动态资源，最后融合双层上下文。

In [4]:
with tempfile.TemporaryDirectory() as tmpdir:
    server, memory = create_memory_enabled_mcp_server(Path(tmpdir) / "e2e.db", window_size=3)
    client = MCPClient(server=server)
    client.initialize()

    client.call_tool(
        "memory_store",
        {
            "key": "release_rule",
            "value": "所有迭代需通过 verify_day.py 与 pytest 校验",
            "namespace": "qa",
            "tags": ["测试", "发布"],
        },
    )
    memory.add_turn("sess-qa", "user", "准备发布 Day 44 版本")
    memory.add_turn("sess-qa", "assistant", "正在检查发布门禁规范...")

    stats = json.loads(client.read_resource("memory://stats")["contents"][0]["text"])
    ctx = memory.build_context("sess-qa", "verify_day.py 发布校验", namespace="qa", top_k=1)
    print("memory://stats ->", stats)
    print("短期记忆轮数:", len(ctx["short_term"]), "| 长期命中 key:", ctx["long_term"][0]["key"])

memory://stats -> {'long_term_count': 1, 'window_size': 3}
短期记忆轮数: 2 | 长期命中 key: release_rule
